# MMLU-TR: trials

Trials before anything is versioned. The goal is to evaluate models on the Turkish translation of MMLU the way Stanford HELM's MMLU leaderboard does.

**HELM's protocol (MMLU leaderboard v1.0.0):**

- 5 example questions per subject from the `dev` split, chosen and ordered by HELM's seeded selection
- a header line, then blocks of `Question:`, options `A.`–`D.` and `Answer:`
- the model generates one token at temperature 0, and the answer is correct when that token is the right letter
- all 14,042 `test` questions

The original MMLU code uses the same layout but scores differently: it takes the most likely of " A", " B", " C" and " D" from the token probabilities.

**Turkish prompts:** the prompt keeps HELM's structure but is fully in Turkish: the header, the `Soru:` and `Cevap:` labels and the subject names. The English version is used only to check the structure against HELM's real prompts.

This notebook compares three methods on the same prompts:

| Method | How the prompt is sent | How the answer is read |
|---|---|---|
| HELM, plain text | As text, without the chat template | The generated letter |
| HELM, chat message | As one user message, thinking off | The generated letter |
| Original MMLU | As text, without the chat template | The most likely of the four letters |

In [1]:
MODEL = "gemma3:4b"
DATASET = "yeniguno/mmlu-tr"
REVISION = "d2c8c04923d7b5a62608ffc58d8e582bc3666e08"
N_SHOTS = 5
# Questions per subject for this trial; the full test split has 14,042.
QUESTIONS_PER_SUBJECT = 2
# Context length; the longest prompt is checked against it below.
NUM_CTX = 8192
OLLAMA_URL = "http://localhost:11434"

## Data

In [2]:
from collections import Counter

from datasets import load_dataset

data = load_dataset(DATASET, revision=REVISION)
dev, test = data["dev"].to_list(), data["test"].to_list()
subjects = sorted({row["subject"] for row in test})

print({split: len(rows) for split, rows in data.items()})
print(f"{len(subjects)} subjects, {set(Counter(row['subject'] for row in dev).values())} dev examples each")

{'dev': 285, 'validation': 1531, 'test': 14042}
57 subjects, {5} dev examples each


## Prompts

`select_shots` is HELM's example selection (`sample_examples`, seed 0). HELM labels each example by the English text of its correct answer, so the English fields decide the order. The Turkish prompt uses the same examples in the same order.

`TEMPLATES` holds HELM's English wording and its Turkish translation. `SUBJECTS_TR` gives each subject's Turkish name.

In [3]:
import random
from collections import defaultdict
from itertools import cycle

LETTERS = "ABCD"


def select_shots(rows: list[dict], k: int = N_SHOTS, seed: int = 0) -> list[dict]:
    """Choose and order a subject's examples exactly as HELM does."""
    random.seed(seed)
    remaining = min(len(rows), k)
    by_label = defaultdict(list)
    for row in rows:
        by_label[row["choices_en"][LETTERS.index(row["answer"])]].append(row)
    labels_by_count = defaultdict(list)
    for label, items in sorted(by_label.items()):
        labels_by_count[len(items)].append(label)
    ordered_labels = []
    for count in sorted(labels_by_count, reverse=True):
        labels = labels_by_count[count]
        random.shuffle(labels)
        ordered_labels.extend(labels)
    shots, labels = [], cycle(ordered_labels)
    while remaining > 0:
        items = by_label[next(labels)]
        if items:
            shots.append(items.pop(random.randrange(len(items))))
            remaining -= 1
    return shots


shots = {subject: select_shots([dict(row) for row in dev if row["subject"] == subject]) for subject in subjects}


TEMPLATES = {
    "tr": {
        "header": "Aşağıdakiler {subject} hakkında çoktan seçmeli sorulardır (cevaplarıyla birlikte).",
        "question": "Soru",
        "answer": "Cevap",
    },
    "en": {
        "header": "The following are multiple choice questions (with answers) about {subject}.",
        "question": "Question",
        "answer": "Answer",
    },
}

SUBJECTS_TR = {
    "abstract_algebra": "soyut cebir",
    "anatomy": "anatomi",
    "astronomy": "astronomi",
    "business_ethics": "iş etiği",
    "clinical_knowledge": "klinik bilgi",
    "college_biology": "üniversite biyolojisi",
    "college_chemistry": "üniversite kimyası",
    "college_computer_science": "üniversite bilgisayar bilimi",
    "college_mathematics": "üniversite matematiği",
    "college_medicine": "üniversite tıbbı",
    "college_physics": "üniversite fiziği",
    "computer_security": "bilgisayar güvenliği",
    "conceptual_physics": "kavramsal fizik",
    "econometrics": "ekonometri",
    "electrical_engineering": "elektrik mühendisliği",
    "elementary_mathematics": "temel matematik",
    "formal_logic": "biçimsel mantık",
    "global_facts": "küresel olgular",
    "high_school_biology": "lise biyolojisi",
    "high_school_chemistry": "lise kimyası",
    "high_school_computer_science": "lise bilgisayar bilimi",
    "high_school_european_history": "lise Avrupa tarihi",
    "high_school_geography": "lise coğrafyası",
    "high_school_government_and_politics": "lise devlet yönetimi ve siyaset",
    "high_school_macroeconomics": "lise makroekonomisi",
    "high_school_mathematics": "lise matematiği",
    "high_school_microeconomics": "lise mikroekonomisi",
    "high_school_physics": "lise fiziği",
    "high_school_psychology": "lise psikolojisi",
    "high_school_statistics": "lise istatistiği",
    "high_school_us_history": "lise ABD tarihi",
    "high_school_world_history": "lise dünya tarihi",
    "human_aging": "insan yaşlanması",
    "human_sexuality": "insan cinselliği",
    "international_law": "uluslararası hukuk",
    "jurisprudence": "hukuk felsefesi",
    "logical_fallacies": "mantık hataları",
    "machine_learning": "makine öğrenmesi",
    "management": "yönetim",
    "marketing": "pazarlama",
    "medical_genetics": "tıbbi genetik",
    "miscellaneous": "çeşitli konular",
    "moral_disputes": "ahlaki tartışmalar",
    "moral_scenarios": "ahlaki senaryolar",
    "nutrition": "beslenme",
    "philosophy": "felsefe",
    "prehistory": "tarih öncesi",
    "professional_accounting": "profesyonel muhasebe",
    "professional_law": "profesyonel hukuk",
    "professional_medicine": "profesyonel tıp",
    "professional_psychology": "profesyonel psikoloji",
    "public_relations": "halkla ilişkiler",
    "security_studies": "güvenlik çalışmaları",
    "sociology": "sosyoloji",
    "us_foreign_policy": "ABD dış politikası",
    "virology": "viroloji",
    "world_religions": "dünya dinleri",
}
assert set(SUBJECTS_TR) == set(subjects), "Every subject needs a Turkish name."


def question_block(template: dict, question: str, choices: list[str], answer: str | None = None) -> str:
    block = f"{template['question']}: {question}\n"
    block += "".join(f"{letter}. {choice}\n" for letter, choice in zip(LETTERS, choices))
    return block + (f"{template['answer']}: {answer}\n" if answer else f"{template['answer']}:")


def build_prompt(subject: str, item: dict, language: str = "tr") -> str:
    """HELM's MMLU prompt for one test question; language="en" gives HELM's English original."""
    template = TEMPLATES[language]
    question, choices = ("question", "choices") if language == "tr" else ("question_en", "choices_en")
    name = SUBJECTS_TR[subject] if language == "tr" else subject.replace("_", " ")
    header = template["header"].format(subject=name) + "\n"
    examples = [question_block(template, shot[question], shot[choices], shot["answer"]) for shot in shots[subject]]
    return "\n".join([header, *examples, question_block(template, item[question], item[choices])])


print(build_prompt("anatomy", next(row for row in test if row["subject"] == "anatomy")))

Aşağıdakiler anatomi hakkında çoktan seçmeli sorulardır (cevaplarıyla birlikte).

Soru: Hiyoid kemiğinin embriyolojik kökeni nedir?
A. Birinci faringeal ark
B. Birinci ve ikinci faringeal arklar
C. İkinci faringeal ark
D. İkinci ve üçüncü faringeal arklar
Cevap: D

Soru: Trigeminal sinirin bu dallarından hangileri somatik motor lifler içerir?
A. Supraorbital sinir
B. İnfraorbital sinir
C. Mental sinir
D. Yukarıdakilerin hiçbiri
Cevap: D

Soru: Aşağıdakilerden hangisi hipofiz bezini içeren vücut boşluğudur?
A. Karın boşluğu
B. Kafatası boşluğu
C. Plevra boşluğu
D. Omurilik boşluğu
Cevap: B

Soru: Plevralarla ilgili aşağıdakilerden hangisi doğrudur?
A. Duyusal innervasyonları yoktur.
B. Birbirlerinden 2 mm'lik bir aralıkla ayrılırlar.
C. Boyuna uzanırlar.
D. Solunum epitelinden oluşurlar.
Cevap: C

Soru: Angle Sınıf II Bölüm 2 oklüzyonunda aşağıdakilerden hangisi görülür?
A. Üst yan kesici dişlerde aşırı dikey örtüşme.
B. Üst orta kesici dişlerde negatif yatay örtüşme.
C. Üst yan kesici 

### Check against HELM

Builds the English version of the prompts for two subjects and compares them with the prompts HELM sent to GPT-4 on its MMLU leaderboard. Every prompt should be identical, which confirms the structure and example order that the Turkish prompts share.

In [4]:
import httpx

HELM_REQUESTS = (
    "https://storage.googleapis.com/crfm-helm-public/mmlu/benchmark_output/runs/v1.0.0/"
    "mmlu:subject={subject},method=multiple_choice_joint,model=openai_gpt-4-0613,"
    "eval_split=test,groups=mmlu_{subject}/display_requests.json"
)


def check_against_helm(subject: str) -> tuple[int, int]:
    """Return (identical prompts, HELM prompts) for one subject."""
    helm_requests = httpx.get(HELM_REQUESTS.format(subject=subject), timeout=120).json()
    by_question = {row["question_en"]: row for row in test if row["subject"] == subject}
    identical = 0
    for request in helm_requests:
        prompt = request["request"]["prompt"]
        item = by_question[prompt.rsplit("Question: ", 1)[1].split("\nA. ")[0]]
        identical += build_prompt(subject, item, language="en") == prompt
    return identical, len(helm_requests)


for subject in ["anatomy", "abstract_algebra"]:
    print(f"{subject}: %d of %d prompts identical" % check_against_helm(subject))

anatomy: 135 of 135 prompts identical


abstract_algebra: 100 of 100 prompts identical


## Asking the model

In [5]:
import time


def ask(prompt: str, chat: bool = False) -> dict:
    """Ask Ollama for one token at temperature 0, with token probabilities.

    chat=False sends the prompt as plain text, without the chat template.
    chat=True sends it as one user message with thinking off.
    """
    body = {
        "model": MODEL,
        "stream": False,
        "options": {"temperature": 0, "num_predict": 1, "num_ctx": NUM_CTX},
        "logprobs": True,
        "top_logprobs": 20,
    }
    if chat:
        body |= {"messages": [{"role": "user", "content": prompt}], "think": False}
    else:
        body |= {"prompt": prompt, "raw": True}
    started = time.perf_counter()
    response = httpx.post(f"{OLLAMA_URL}/api/{'chat' if chat else 'generate'}", json=body, timeout=600).json()
    if "error" in response:
        raise RuntimeError(response["error"])
    top = response["logprobs"][0]["top_logprobs"]
    return {
        "text": response["message"]["content"] if chat else response["response"],
        # The original MMLU code reads " A" to " D" and gives a missing letter -100.
        "letter_logprobs": {letter: next((t["logprob"] for t in top if t["token"] == f" {letter}"), -100.0) for letter in LETTERS},
        "prompt_tokens": response["prompt_eval_count"],
        "seconds": time.perf_counter() - started,
    }


item = test[0]
print(item["id"], "correct answer:", item["answer"])
print("plain text:", ask(build_prompt(item["subject"], item)))
print("chat:", ask(build_prompt(item["subject"], item), chat=True))

abstract_algebra_test_0001 correct answer: B


plain text: {'text': ' B', 'letter_logprobs': {'A': -4.6668925285339355, 'B': -0.029287004843354225, 'C': -4.178828716278076, 'D': -5.647000789642334}, 'prompt_tokens': 458, 'seconds': 0.07834601700005805}
chat: {'text': 'C', 'letter_logprobs': {'A': -100.0, 'B': -100.0, 'C': -100.0, 'D': -100.0}, 'prompt_tokens': 468, 'seconds': 0.0726584090000415}


### Longest prompt

Sends the longest test prompt once to check that it fits in `NUM_CTX`. Ollama cuts longer prompts without an error.

In [6]:
longest = max(test, key=lambda row: len(build_prompt(row["subject"], row)))
tokens = ask(build_prompt(longest["subject"], longest))["prompt_tokens"]
print(f"{longest['id']}: {tokens} tokens of {NUM_CTX}")
assert tokens < NUM_CTX, "Increase NUM_CTX: the longest prompt doesn't fit."

high_school_european_history_test_0143: 4039 tokens of 8192


## Trial

In [7]:
import pandas as pd

sample = [row for subject in subjects for row in [r for r in test if r["subject"] == subject][:QUESTIONS_PER_SUBJECT]]
records = []
for position, item in enumerate(sample, start=1):
    prompt = build_prompt(item["subject"], item)
    plain, chat = ask(prompt), ask(prompt, chat=True)
    records.append({
        "id": item["id"],
        "subject": item["subject"],
        "answer": item["answer"],
        "helm_plain": plain["text"].strip(),
        "helm_chat": chat["text"].strip(),
        "original": max(LETTERS, key=plain["letter_logprobs"].get),
        "prompt_tokens": plain["prompt_tokens"],
        "seconds_plain": plain["seconds"],
        "seconds_chat": chat["seconds"],
    })
    if position % 20 == 0:
        print(f"{position} / {len(sample)}")
results = pd.DataFrame(records)
print(f"{len(results)} questions from {results.subject.nunique()} subjects")

20 / 114


40 / 114


60 / 114


80 / 114


100 / 114


114 questions from 57 subjects


In [8]:
METHODS = {"HELM, plain text": "helm_plain", "HELM, chat message": "helm_chat", "Original MMLU": "original"}
pd.DataFrame([
    {
        "Method": name,
        "Accuracy": (results[column] == results.answer).mean(),
        "Invalid answers": (~results[column].isin(list(LETTERS))).mean(),
        "Median seconds": results["seconds_chat" if column == "helm_chat" else "seconds_plain"].median(),
    }
    for name, column in METHODS.items()
]).style.format({"Accuracy": "{:.2%}", "Invalid answers": "{:.2%}", "Median seconds": "{:.2f}"})

,Method,Accuracy,Invalid answers,Median seconds
0,"HELM, plain text",53.51%,0.00%,0.10
1,"HELM, chat message",34.21%,9.65%,0.10
2,Original MMLU,53.51%,0.00%,0.10


In [9]:
print("Plain-text letter matches the most likely letter:", f"{(results.helm_plain == results.original).mean():.2%}")
print("Most common chat outputs:", results.helm_chat.value_counts().head(5).to_dict())

Plain-text letter matches the most likely letter: 100.00%
Most common chat outputs: {'C': 81, 'A': 8, 'D': 8, 'Har': 6, 'B': 6}


## Still to decide

- **Method:** which of the three to use as the main score
- **Context length:** keep `NUM_CTX` only if the longest prompt fits
- **Models:** which models to run

After that, the chosen setup is versioned like the university entrance benchmark.